# SimCLR on Oxford-IIIT Pets

This notebook implements:
- Self-supervised pretraining using SimCLR  
- Fine-tuning for 10-class classification  
- Evaluation on test set


# Data Augmentation (SimCLR)

SimCLR relies heavily on strong augmentations:
- Random crop
- Color jitter
- Blur
- Flip

These define what the model considers "similar".

## SimCLR Model

- Encoder: ResNet18
- Projection head: MLP
- Output normalized embeddings

## Pretraining

Train encoder using contrastive loss on unlabeled data.

## Contrastive Learning (SimCLR)

SimCLR learns representations by bringing two augmented views of the same image closer while pushing other samples apart.

### NT-Xent (Normalized Temperature-Scaled Cross-Entropy) Loss
$$
\ell(i, j) = -\log \frac{\exp(\mathrm{sim}(z_i, z_j) / \tau)}{\sum_{k \neq i} \exp(\mathrm{sim}(z_i, z_k) / \tau)}
$$
Where:
- $$(z_i, z_j)$$ are embeddings of two augmented views
- $$\mathrm{sim}(\cdot)$$ is cosine similarity
- $$\tau$$ is temperature

In [1]:
!pip install torch torchvision torchaudio scikit-learn timm transformers vit_pytorch

In [2]:
# -*- coding: utf-8 -*-
"""
Created on Thu Mar 20 13:37:08 2025

@author: vigo
"""
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models

from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision.datasets import STL10

device = "cuda" if torch.cuda.is_available() else "cpu"


#Check for CUDA and use if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
# Load full STL10 dataset
pretrain_dataset = STL10(root="./data", split="unlabeled", download=True)

finetune_dataset = STL10(root="./data", split="train", download=True)

test_dataset = STL10(root="./data", split="test", download=True)


# Transform for unlabeled data 
simclr_transform = transforms.Compose([
    transforms.RandomResizedCrop(96),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([
        transforms.ColorJitter(0.8, 0.8, 0.8, 0.2)
    ], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.GaussianBlur(kernel_size=9),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])


# pretained Dataset Wrapper
class SimCLRDataset(Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        x_i = self.transform(img)
        x_j = self.transform(img)
        return x_i, x_j





# Transform for labelled data (finetune and test data)
supervised_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# Finetuning and test datasets 
class SupervisedDataset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        img = self.transform(img)
        return img, label



# Create loaders
batch_size = 128

#pretrain data loader
simclr_loader = DataLoader(
    SimCLRDataset(pretrain_dataset, simclr_transform),
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)


#finetune and test data loaders
train_loader = DataLoader(
    SupervisedDataset(finetune_dataset, supervised_transform),
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    SupervisedDataset(test_dataset, supervised_transform),
    batch_size=batch_size,
    shuffle=False
)

In [ ]:

# Definition of SimCLR (pretrained) Model
class SimCLR(nn.Module):
    def __init__(self, dim=128):
        super().__init__()

        self.encoder = models.resnet18(weights=None)
        self.encoder.fc = nn.Identity()

        self.projector = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return F.normalize(z, dim=1)


# NT-Xent Loss
def nt_xent_loss(z1, z2, temp=0.1):
    N = z1.size(0)

    z = torch.cat([z1, z2], dim=0)
    z = F.normalize(z, dim=1)

    sim = torch.matmul(z, z.T) / temp

    mask = torch.eye(2*N, dtype=torch.bool).to(z.device)
    sim = sim.masked_fill(mask, -1e9)

    positives = torch.cat([
        torch.arange(N, 2*N),
        torch.arange(0, N)
    ]).to(z.device)

    return F.cross_entropy(sim, positives)


# Pretraining

model = SimCLR().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for epoch in range(20):
    model.train()

    for x_i, x_j in simclr_loader:
        x_i, x_j = x_i.to(device), x_j.to(device)

        z_i = model(x_i)
        z_j = model(x_j)

        loss = nt_xent_loss(z_i, z_j)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Pretrain Epoch {epoch} Loss {loss.item():.4f}")

torch.save(model.encoder.state_dict(), "simclr_model_stl10.pth")

Pretrain Epoch 0 Loss 3.6122


In [ ]:

# adapting the pretrained models to actual task (i.e., 10-class classification)

class Classifier(nn.Module):
    def __init__(self, encoder, num_classes=10):
        super().__init__()
        self.encoder = encoder
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        return self.fc(self.encoder(x))


# Load saved pretrained model

encoder = models.resnet18(weights=None)
encoder.fc = nn.Identity()
encoder.load_state_dict(torch.load("simclr_model_stl10.pth", map_location=device))

model = Classifier(encoder).to(device)

for p in model.encoder.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam([
    {"params": model.encoder.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-4}
])
criterion = nn.CrossEntropyLoss()

#Finetuning training

for epoch in range(10):
    model.train()

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        loss = criterion(model(x), y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Finetune Epoch {epoch} Loss {loss.item():.4f}")

# Evaluation on testset

model.eval()
correct, total = 0, 0

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)

        preds = model(x).argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

print("Test Accuracy:", correct / total)
